<a href="https://colab.research.google.com/github/Amonstarr/Twitter-Classification-Using-Indobert/blob/main/Indobert_preprocessing_engine_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 8.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Satria Data/dataset_penyisihan_bdc_2024 (2).csv", sep=';')
df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [ ]:
df_test = pd.read_csv("/content/drive/MyDrive/Satria Data/dataset_unlabeled_penyisihan_bdc_2024.csv", sep=';')
df_test.head()

,IDText,Text
0,TXT0001,Lu mau org2 pro-demokrasi di negara ini bisa p...
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...
2,TXT0003,kiki_daliyo Ganjar Pranowo itulah beliau soso...
3,TXT0004,@kumparan Prabowo Gibran yang bisa melakukan i...
4,TXT0005,@sniperruben45 @uda_zulhendra @ainunnajib Lah ...


In [ ]:
KOLOM_LABEL= "label" #@param {type:"string"}
print(df[KOLOM_LABEL].unique())
# print(df_test[KOLOM_LABEL].unique())


['Sumber Daya Alam' 'Politik' 'Demografi' 'Pertahanan dan Keamanan'
 'Ideologi' 'Ekonomi' 'Sosial Budaya' 'Geografi']


# Preprocessing

In [ ]:
class DataCleaning:
  # Initialization
  factory     = StemmerFactory()
  stemmer     = factory.create_stemmer()
  kamus_alay1 = pd.read_csv('https://raw.githubusercontent.com/fendiirfan/Kamus-Alay/main/Kamus-Alay.csv') # Corrected URL
  kamus_alay1 = kamus_alay1.set_index('kataAlay')
  kamus_alay2 = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv')
  kamus_alay2 = kamus_alay2.filter(['slang', 'formal'], axis=1)
  kamus_alay2 = kamus_alay2.drop_duplicates(subset=['slang'], keep='first')
  kamus_alay2 = kamus_alay2.set_index('slang')
  stopword1   = list(pd.read_csv('https://raw.githubusercontent.com/datascienceid/stopwords-bahasa-indonesia/master/stopwords_id_satya.txt', header = None)[0])
  custom_word = []

  @classmethod
  def CleanDataFrame(cls, df, col_name, label_name, jum_minimum=None, minimum_kata=0, label_mapping=None, dropna=False):

    final_list_clean = []
    final_list_kotor = []
    final_label = []

    if jum_minimum == None: jum_minimum = len(df)
    if len(df) < jum_minimum: raise "Jumlah Data Yang Diinginkan melebihi Data yang Ada"
    i = 0
    current = 0

    while i < len(df):
      current_kalimat = df.loc[i][col_name]
      current_label = df.loc[i][label_name]

      clean_kalimat = cls.__cleanSentence__(current_kalimat)
      if type(clean_kalimat) != str or clean_kalimat == None or clean_kalimat == "":
        print("Ditemukan string kosong setelah di preprocessed pada kalimat: ", current_kalimat)
        clean_kalimat = "NaN"
      if (len(clean_kalimat.split(' ')) >= minimum_kata):
        final_list_clean.append(str(clean_kalimat))
        final_list_kotor.append(str(current_kalimat))
        if label_mapping != None:
          final_label.append(label_mapping[current_label])
        else:
          final_label.append(current_label)
        current += 1

        if current % 10 == 0:
          print("Memproses {} data".format(current))

      if current == jum_minimum:
        break

      i += 1

    data = {
        'raw': final_list_kotor,
        'processed': final_list_clean,
        'label': final_label
    }

    final_df = pd.DataFrame(data)
    if dropna:
      print("NaN Dropped")
      final_df = final_df.dropna(how='any')

    final_df['processed'] = final_df['processed'].astype(str)
    final_df['raw'] = final_df['raw'].astype(str)
    return final_df

  @classmethod
  def __cleanSentence__(cls, text):
    '''
    Melakukan prapemrosesan pada suatu kalimat dengan menghilangkan formatting pada kalimat,
    menghilangkan stopword pada kalimat, mengganti kata alay yang sudah terdefinisikan, serta
    melakukan stemming kalimat tersebut.
    '''

    # #
    # Cleaning Formatted Text Link And Tag (@username) using Regex
    # #
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(@\w+|#\w+)','',text)

    #will replace the html characters with " "
    text = re.sub('<.*?>', '', text)

    temp_text = list(text)
    for i in range(len(temp_text)):
      if temp_text[i] in string.punctuation:
        temp_text[i] = " "

    text = ''.join(temp_text)

    #will consider only alphabets
    text = re.sub('[^a-zA-Z]',' ',text)
    #will replace newline with space
    text = re.sub("\n"," ",text)
    #will convert to lower case
    text = text.lower()
    # will replace a word
    text = re.sub(r"(username|user|url|rt|xf|fx|xe|xa)\s|\s(user|url|rt|xf|fx|xe|xa)","",text)
    # will repalce repated char
    text = re.sub(r'(\w)(\1{2,})', r"\1", text)
    # will replace single word
    text = re.sub(r"\b[a-zA-Z]\b","",text)
    # will replace space more than one
    text = re.sub('(s{2,})',' ',text)
    # will join the words
    text=' '.join(text.split())

    text_split = text.split(' ')
    # #
    # Mengganti kata-kata yang tidak baku
    # aku gapakai try catch lagi, lebih simple malah ini
    # #
    for i in range(len(text_split)):
      if text_split[i] in cls.kamus_alay1.index:
        text_split[i] = cls.kamus_alay1.loc[text_split[i]]['kataBaik']
      elif text_split[i] in cls.kamus_alay2.index:
        text_split[i] = cls.kamus_alay2.loc[text_split[i]]['formal']

    # #
    # Stemming
    # #
    stemmed_text = cls.stemmer.stem(text)

    # #
    # Removing Stopwords and custom word
    # #
    temp_text_split = []
    for i in range(len(text_split)):
      if (text_split[i] not in cls.stopword1) and (text_split[i] not in cls.custom_word) and (type(text_split[i]) == str):
        temp_text_split.append(text_split[i])

    # Memastikan saja
    if len(temp_text_split) == 0:
      return ""
    else:
      final_text = ' '.join(temp_text_split)

    return final_text

Non-Stopword


In [ ]:
# class DataCleaning:
#   # Initialization
#   factory     = StemmerFactory()
#   stemmer     = factory.create_stemmer()
#   kamus_alay1 = pd.read_csv('https://raw.githubusercontent.com/fendiirfan/Kamus-Alay/main/Kamus-Alay.csv') # Corrected URL
#   kamus_alay1 = kamus_alay1.set_index('kataAlay')
#   kamus_alay2 = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv')
#   kamus_alay2 = kamus_alay2.filter(['slang', 'formal'], axis=1)
#   kamus_alay2 = kamus_alay2.drop_duplicates(subset=['slang'], keep='first')
#   kamus_alay2 = kamus_alay2.set_index('slang')
#   stopword1   = list(pd.read_csv('https://raw.githubusercontent.com/datascienceid/stopwords-bahasa-indonesia/master/stopwords_id_satya.txt', header = None)[0])
#   custom_word = []

#   @classmethod
#   def CleanDataFrame(cls, df, col_name, label_name, jum_minimum=None, minimum_kata=0, label_mapping=None, dropna=False):

#     final_list_clean = []
#     final_list_kotor = []
#     final_label = []

#     if jum_minimum == None: jum_minimum = len(df)
#     if len(df) < jum_minimum: raise "Jumlah Data Yang Diinginkan melebihi Data yang Ada"
#     i = 0
#     current = 0

#     while i < len(df):
#       current_kalimat = df.loc[i][col_name]
#       current_label = df.loc[i][label_name]

#       clean_kalimat = cls.__cleanSentence__(current_kalimat)
#       if type(clean_kalimat) != str or clean_kalimat == None or clean_kalimat == "":
#         print("Ditemukan string kosong setelah di preprocessed pada kalimat: ", current_kalimat)
#         clean_kalimat = "NaN"
#       if (len(clean_kalimat.split(' ')) >= minimum_kata):
#         final_list_clean.append(str(clean_kalimat))
#         final_list_kotor.append(str(current_kalimat))
#         if label_mapping != None:
#           final_label.append(label_mapping[current_label])
#         else:
#           final_label.append(current_label)
#         current += 1

#         if current % 10 == 0:
#           print("Memproses {} data".format(current))

#       if current == jum_minimum:
#         break

#       i += 1

#     data = {
#         'raw': final_list_kotor,
#         'processed': final_list_clean,
#         'label': final_label
#     }

#     final_df = pd.DataFrame(data)
#     if dropna:
#       print("NaN Dropped")
#       final_df = final_df.dropna(how='any')

#     final_df['processed'] = final_df['processed'].astype(str)
#     final_df['raw'] = final_df['raw'].astype(str)
#     return final_df

#   @classmethod
#   def __cleanSentence__(cls, text):
#     '''
#     Melakukan prapemrosesan pada suatu kalimat dengan menghilangkan formatting pada kalimat,
#     menghilangkan stopword pada kalimat, mengganti kata alay yang sudah terdefinisikan, serta
#     melakukan stemming kalimat tersebut.
#     '''

#     # #
#     # Cleaning Formatted Text Link And Tag (@username) using Regex
#     # #
#     text = re.sub(r'http\S+', '', text)
#     text = re.sub('(@\w+|#\w+)','',text)

#     #will replace the html characters with " "
#     text = re.sub('<.*?>', '', text)

#     temp_text = list(text)
#     for i in range(len(temp_text)):
#       if temp_text[i] in string.punctuation:
#         temp_text[i] = " "

#     text = ''.join(temp_text)

#     #will consider only alphabets
#     text = re.sub('[^a-zA-Z]',' ',text)
#     #will replace newline with space
#     text = re.sub("\n"," ",text)
#     #will convert to lower case
#     text = text.lower()
#     # will replace a word
#     text = re.sub("(username|user|url|rt|xf|fx|xe|xa)\s|\s(user|url|rt|xf|fx|xe|xa)","",text)
#     # will repalce repated char
#     text = re.sub(r'(\w)(\1{2,})', r"\1", text)
#     # will replace single word
#     text = re.sub(r"\b[a-zA-Z]\b","",text)
#     # will replace space more than one
#     text = re.sub('(s{2,})',' ',text)
#     # will join the words
#     text=' '.join(text.split())

#     text_split = text.split(' ')
#     # #
#     # Mengganti kata-kata yang tidak baku
#     # aku gapakai try catch lagi, lebih simple malah ini
#     # #
#     for i in range(len(text_split)):
#       if text_split[i] in cls.kamus_alay1.index:
#         text_split[i] = cls.kamus_alay1.loc[text_split[i]]['kataBaik']
#       elif text_split[i] in cls.kamus_alay2.index:
#         text_split[i] = cls.kamus_alay2.loc[text_split[i]]['formal']

#     # #
#     # Stemming
#     # #
#     stemmed_text = cls.stemmer.stem(text)

#     # #
#     # Removing Stopwords and custom word
#     # #
#     temp_text_split = []
#     for i in range(len(text_split)):
#       if (type(text_split[i]) == str):
#         temp_text_split.append(text_split[i])

#     # Memastikan saja
#     if len(temp_text_split) == 0:
#       return ""
#     else:
#       final_text = ' '.join(temp_text_split)

#     return final_text

In [ ]:
unique_labels = df[KOLOM_LABEL].unique()
label_mapping = {label: i for i, label in enumerate(unique_labels)}

print("Generated label_mapping:")
print(label_mapping)

Generated label_mapping:
{'Sumber Daya Alam': 0, 'Politik': 1, 'Demografi': 2, 'Pertahanan dan Keamanan': 3, 'Ideologi': 4, 'Ekonomi': 5, 'Sosial Budaya': 6, 'Geografi': 7}


In [ ]:
df = DataCleaning.CleanDataFrame(df, 'text', 'label', None, 2, label_mapping, dropna=True)
print(len(df))

Memproses 10 data
Memproses 20 data
Memproses 30 data
Memproses 40 data
Memproses 50 data
Memproses 60 data
Memproses 70 data
Memproses 80 data
Memproses 90 data
Memproses 100 data
Memproses 110 data
Memproses 120 data
Memproses 130 data
Memproses 140 data
Memproses 150 data
Memproses 160 data
Memproses 170 data
Memproses 180 data
Memproses 190 data
Memproses 200 data
Memproses 210 data
Memproses 220 data
Memproses 230 data
Memproses 240 data
Memproses 250 data
Memproses 260 data
Memproses 270 data
Memproses 280 data
Memproses 290 data
Memproses 300 data
Memproses 310 data
Memproses 320 data
Memproses 330 data
Memproses 340 data
Memproses 350 data
Memproses 360 data
Memproses 370 data
Memproses 380 data
Memproses 390 data
Memproses 400 data
Memproses 410 data
Memproses 420 data
Memproses 430 data
Memproses 440 data
Memproses 450 data
Memproses 460 data
Memproses 470 data
Memproses 480 data
Memproses 490 data
Memproses 500 data
Memproses 510 data
Memproses 520 data
Memproses 530 data
Me

In [ ]:
# Jangan Ditimpa Data Aslinya
df.to_csv(
    '/content/drive/MyDrive/Satria Data/dataset_penyisihan_bdc_2024_clean.csv',
    index=False,
    encoding='utf-8',
    na_rep=""
)
df[df['processed'] == "NaN"]

,raw,processed,label


In [ ]:
df_test['processed'] = df_test['Text'].apply(
    DataCleaning.__cleanSentence__
)

#Jangan Ditimpa Data Aslinya
df_test.to_csv('/content/drive/MyDrive/Satria Data/datatest_penyisihan_bdc_2024_clean.csv', index=False)